# 01 · Update the project environment

Reports what is out of date and (when you tell it to) upgrades the packages in `.venv/`.

**Safe by default:** `DRY_RUN = True` only reports. Nothing changes until you set it to `False`.

**What it can't do:** change the Python version. To move to a newer Python, install it, open `00_setup_environment.ipynb`, set `RECREATE_VENV = True`, and run it.

It works on the project's `.venv/` from outside, so it can run from any kernel. Restart your analysis notebooks' kernels afterwards so they pick up the new versions.

In [ ]:
# ---- Settings (edit here) ----
VENV_DIR = ".venv"
DRY_RUN = True              # True = only report what is outdated. False = actually upgrade.
WRITE_LOCK_FILE = True      # after upgrading, record exact versions in requirements-lock.txt
RECOMMENDED_PYTHON = (3, 12)

In [ ]:
import json
import platform
import re
import subprocess
from pathlib import Path

IS_WINDOWS = platform.system() == "Windows"


def find_repo_root(start=None):
    here = Path(start or Path.cwd()).resolve()
    for folder in [here] + list(here.parents):
        if (folder / "requirements.txt").exists():
            return folder
    raise FileNotFoundError("Could not find requirements.txt. Open this notebook from inside the project folder.")


ROOT = find_repo_root()
venv_python = ROOT / VENV_DIR / ("Scripts/python.exe" if IS_WINDOWS else "bin/python")
if not venv_python.exists():
    raise FileNotFoundError("No %s found. Run 00_setup_environment.ipynb first." % VENV_DIR)


def run(cmd, show=True, tail=None):
    cmd = [str(c) for c in cmd]
    shown = [c if "\n" not in c else "<code>" for c in cmd]
    print("$", " ".join(shown))
    result = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)
    if show and result.stdout.strip():
        lines = result.stdout.strip().splitlines()
        print("\n".join(lines[-tail:] if tail else lines))
    if result.returncode != 0:
        print(result.stderr.strip() or result.stdout.strip())
        raise RuntimeError("Command failed (exit code %d)." % result.returncode)
    return result


print("Project root:", ROOT)

## 1 · What is installed now

In [ ]:
text = run([venv_python, "-c", "import sys; print(sys.version.split()[0])"], show=False).stdout.strip()
env_version = tuple(int(p) for p in text.split(".")[:2])
print("Environment Python:", text)

if env_version < RECOMMENDED_PYTHON:
    print(
        "\nNOTE: this environment's Python is older than the recommended %d.%d. Packages can still be upgraded here,"
        "\nbut newer package versions may not support it. To move Python itself: install a newer Python, then run"
        "\n00_setup_environment.ipynb with RECREATE_VENV = True." % RECOMMENDED_PYTHON
    )

def requirement_names(path):
    names = []
    for line in path.read_text().splitlines():
        line = line.split("#")[0].strip()
        if line:
            names.append(re.split(r"[<>=!~\[; ]", line, maxsplit=1)[0].lower().replace("_", "-"))
    return names

direct = requirement_names(ROOT / "requirements.txt")
print("\nProject packages:", ", ".join(direct))

## 2 · What is out of date

In [ ]:
out = run([venv_python, "-m", "pip", "list", "--outdated", "--format=json"], show=False)
outdated = json.loads(out.stdout or "[]")

mine = [p for p in outdated if p["name"].lower().replace("_", "-") in direct]
others = [p for p in outdated if p not in mine]

if mine:
    print("%-16s %-12s %s" % ("package", "installed", "latest"))
    for p in mine:
        print("%-16s %-12s %s" % (p["name"], p["version"], p["latest_version"]))
else:
    print("All project packages are up to date.")
if others:
    print("\n(%d other dependency package(s) are also behind; they update along with the ones above.)" % len(others))

print(
    "\nNote: upgrades stay inside the ranges in requirements.txt. To cap a package (for example pandas<3), edit that file."
)

## 3 · Upgrade (only when `DRY_RUN = False`)

In [ ]:
if DRY_RUN:
    print("DRY_RUN is True, so nothing was changed. Set DRY_RUN = False at the top and re-run from here to upgrade.")
else:
    print("Upgrading...")
    run([venv_python, "-m", "pip", "install", "--progress-bar", "off", "--no-color", "--disable-pip-version-check", "--upgrade", "pip"], tail=2)
    run([venv_python, "-m", "pip", "install", "--progress-bar", "off", "--no-color", "--disable-pip-version-check", "--upgrade", "-r", "requirements.txt"], tail=4)

## 4 · Record exact versions

In [ ]:
if DRY_RUN:
    print("Skipped (DRY_RUN is True).")
elif WRITE_LOCK_FILE:
    frozen = run([venv_python, "-m", "pip", "freeze"], show=False).stdout
    header = "# Generated by 01_update_environment.ipynb. Exact versions that last worked together.\n"
    (ROOT / "requirements-lock.txt").write_text(header + frozen)
    print("Wrote requirements-lock.txt (%d packages)." % len(frozen.strip().splitlines()))
else:
    print("WRITE_LOCK_FILE is False, so no lock file was written.")

## 5 · Verify

In [ ]:
check = '''
import importlib, sys
print("Python", sys.version.split()[0])
for name in ["pandas", "requests", "dotenv", "ipykernel"]:
    module = importlib.import_module(name)
    print("%-10s %s" % (name, getattr(module, "__version__", "ok")))
'''
run([venv_python, "-c", check])
print("\nRestart any open notebook kernels so they pick up the new versions.")